# Michigan Traders: Module 6
# Backtesting Mechanics & Performance Analytics  ·  *ANSWER KEY*

**Series:** MAT Education · Evaluation & Research
**Level:** Intermediate (builds on Modules 1–5)
**Format:** **Guided + Your-Turn**, with self-checking exercises you can run on your own laptop.

---

Module 5 ended with a strategy and no way to judge it. This module supplies the judgement.

Two halves:

1. **Mechanics**: what QuantConnect's backtester actually simulates when you call `set_holdings`. Fills, slippage, fees, cash settlement, and order timing. Most "my backtest looked great and live trading did not" stories are a mechanics problem.
2. **Analytics**: every statistic on the QuantConnect results page, what it means, and how to compute it yourself. You will implement Sharpe, Sortino, drawdown, beta, alpha, win rate and profit factor in pandas, so that the numbers stop being magic.

Then the part that matters most and gets taught least: **how to not fool yourself**. A backtest is a hypothesis test you are running on data you have already seen, which makes it very easy to get a beautiful answer that means nothing.

## How to use this notebook

| Cell type | Where it runs | What to do |
|---|---|---|
| 🟢 **Local cell** | your laptop's Jupyter | Run it. Output is baked in so you can read along. |
| 🔵 **QC cell** | QuantConnect (LEAN) | Copy into an algorithm project. It will *not* run locally. |

The analytics half runs entirely on your laptop, on a synthetic strategy return series. That is deliberate: every statistic QuantConnect reports is computable from a return series, and computing them yourself once is the fastest way to stop misreading them.

Answers are in `06_Backtesting_Mechanics_and_Performance_SOLUTIONS.ipynb`.

### Setup: a strategy, a benchmark, and a trade log

We build three things:

- `bench_ret`: daily returns of a buy-and-hold benchmark (think SPY).
- `strat_ret`: daily returns of a trend-following strategy that is in the market some of the time. It is built from a real signal on a real (synthetic) price series, so its statistics behave like a real strategy's: fat-ish tails, drawdowns, and stretches of doing nothing.
- `position`: the strategy's daily position (1.0 invested, 0.0 flat), which we need for turnover and cost work.

The price series again uses stitched regimes so the trend follower has something to follow, and the RNG is seeded.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)

rng = np.random.default_rng(85)
dates = pd.bdate_range("2020-01-02", periods=1008)     # ~4 trading years

# Six stitched regimes so there is genuine in-sample / out-of-sample variety.
regimes = [(168, 0.0016, 0.008), (168, -0.0010, 0.016), (168, 0.0005, 0.009),
           (168, 0.0018, 0.008), (168, -0.0013, 0.018), (168, 0.0010, 0.009)]
daily = np.concatenate([rng.normal(mu, sd, n) for n, mu, sd in regimes])

price = pd.Series(100 * np.exp(np.cumsum(daily)), index=dates, name="close")
bench_ret = price.pct_change()

# A 20/60 crossover with a 150-day trend filter, shifted to avoid look-ahead.
fast, slow = price.rolling(20).mean(), price.rolling(60).mean()
trend = price.rolling(150).mean()
position = ((fast > slow) & (price > trend)).astype(float)

strat_ret = (position.shift(1) * bench_ret).dropna()
bench_ret = bench_ret.loc[strat_ret.index]
position = position.loc[strat_ret.index]

print("sample:", strat_ret.index[0].date(), "->", strat_ret.index[-1].date(),
      f"({len(strat_ret)} days)")
print("invested on", f"{position.mean():.1%}", "of days")
pd.DataFrame({"strategy": strat_ret, "benchmark": bench_ret, "position": position}).head(3)

sample: 2020-01-03 -> 2023-11-13 (1007 days)
invested on 56.4% of days


,strategy,benchmark,position
2020-01-03,-0.0,-0.0073,0.0
2020-01-06,-0.0,-0.0058,0.0
2020-01-07,-0.0,-0.0058,0.0


Note what `strat_ret` is: on days the strategy is flat, its return is exactly `0.0`, not `NaN`. That distinction matters for every statistic below. A flat day is a real day on which you made nothing, and it belongs in the denominator when you compute an average.

## Part 1: Mechanics

## 1. What actually happens when you call `set_holdings`

In Module 3 you wrote `self.set_holdings(self.symbol, 1.0)` and a position appeared. Here is what QuantConnect did in between.

```
  on_data fires with today's bar
        │
        ▼
  set_holdings(symbol, 1.0)
        │
        ├─► compute target quantity from portfolio value and current price
        ├─► create a MARKET ORDER for the difference
        ▼
  the order sits until the NEXT tradable price
        │
        ├─► fill model decides the fill price
        ├─► slippage model adjusts it against you
        ├─► fee model charges commission
        ▼
  portfolio updated: cash down, holdings up
```

The important line is **"the order sits until the next tradable price"**. With daily data, an order placed on today's bar fills at the *next* bar. You do not get today's close. That is the engine enforcing the causality Module 5 warned you about, and it is why a QuantConnect backtest is harder to fool than a pandas one.

### The three reality models

| Model | What it decides | Default |
|---|---|---|
| **Fill model** | *whether* and *at what price* an order fills | immediate fill at market price |
| **Slippage model** | how much the price moves against you | a small constant, brokerage-dependent |
| **Fee model** | commission charged | brokerage-dependent, often per-share |

You set all three at once by choosing a brokerage model, which is the realistic thing to do:

```python
self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE, AccountType.MARGIN)
```

That one line gives you Interactive Brokers' actual fee schedule, their fill behaviour, and margin rules. Without it you get QuantConnect's defaults, which are reasonable but generic.

In [ ]:
# 🔵 QC cell, a backtest configured to be honest
class RealisticBacktest(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2020, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)

        # Model a real broker: real fees, real fills, real margin rules.
        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)

        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        self.fast = self.sma(self.symbol, 20, Resolution.DAILY)
        self.slow = self.sma(self.symbol, 60, Resolution.DAILY)
        self.set_warm_up(70, Resolution.DAILY)

    def on_data(self, data: Slice):
        if self.is_warming_up or not self.slow.is_ready:
            return
        if not data.bars.contains_key(self.symbol):
            return

        long_now = self.fast.current.value > self.slow.current.value
        invested = self.portfolio[self.symbol].invested

        if long_now and not invested:
            self.set_holdings(self.symbol, 1.0)
        elif not long_now and invested:
            self.liquidate(self.symbol)

### Cash settlement and the leverage trap

`set_holdings(symbol, 1.0)` means "target 100% of **portfolio value** in this symbol". Two consequences people trip over:

- `set_holdings(a, 0.6)` then `set_holdings(b, 0.6)` targets **120%** of your portfolio, which requires margin. With a cash account it will fail to fill; with a margin account it will silently lever you up.
- Targets are recomputed from *current* portfolio value, so as your equity grows the same `1.0` target means more shares. This is compounding, and it is usually what you want.

To size in shares rather than fractions, use `self.market_order(symbol, quantity)`. To go flat, `self.liquidate(symbol)`.

### The order types, and which one lies to you

Module 3 used `set_holdings` and `market_order`. Here is the full set, because the choice changes what the backtest is actually claiming.

| Method | Fill guaranteed? | Price guaranteed? | Use when |
|---|---|---|---|
| `self.market_order(sym, qty)` | ✅ yes | ❌ no | you must be in or out now |
| `self.limit_order(sym, qty, limit_price)` | ❌ no | ✅ yes (or better) | price matters more than certainty |
| `self.stop_market_order(sym, qty, stop_price)` | ✅ once triggered | ❌ no | protective stop-loss |
| `self.stop_limit_order(sym, qty, stop, limit)` | ❌ no | ✅ once triggered | stop with a price floor |
| `self.set_holdings(sym, fraction)` | ✅ yes | ❌ no | portfolio-weight targeting |
| `self.liquidate(sym)` | ✅ yes | ❌ no | go flat |

A negative `qty` sells. `self.liquidate()` with no argument closes everything.

> ⚠️ **Limit orders are where backtests lie most.** LEAN fills your limit order if the price *touched* your limit. In live trading, the price touching your limit means you were somewhere in a queue of orders at that price, and you may well not have been filled, especially when the touch was brief, which is exactly when the trade would have been good. A backtest built on limit fills is systematically optimistic. If a strategy only works with limit orders, be suspicious of it.

### Watching your orders fill

`on_order_event` is a sixth method QuantConnect calls, alongside the five pillars from Module 3. It fires every time an order changes state, which is how you confirm the engine did what you meant.

In [ ]:
# 🔵 QC cell, order types and the fill callback
class OrderMechanics(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2023, 1, 1)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.entry_price = None

    def on_data(self, data: Slice):
        if not data.bars.contains_key(self.symbol):
            return
        price = data.bars[self.symbol].close

        if not self.portfolio[self.symbol].invested:
            # Buy now, at whatever the next price turns out to be.
            self.market_order(self.symbol, 100)
            self.entry_price = price

            # Protective stop 5% below entry. Triggers a market order if touched.
            self.stop_market_order(self.symbol, -100, price * 0.95)

    def on_order_event(self, order_event: OrderEvent):
        # Called on every submit, fill, partial fill and cancellation.
        if order_event.status != OrderStatus.FILLED:
            return

        order = self.transactions.get_order_by_id(order_event.order_id)
        self.debug(f"{self.time.date()} {order.type} "
                   f"{order_event.fill_quantity} @ {order_event.fill_price:.2f} "
                   f"fee {order_event.order_fee}")

Three things to notice.

`order_event.status` cycles through `SUBMITTED`, `PARTIALLY_FILLED`, `FILLED`, `CANCELED`, `INVALID`. Only `FILLED` means money moved, so guard on it.

`order_event.fill_price` is the price you actually got, which is *not* the price you saw in `on_data`. Logging both is the fastest way to see slippage in your own backtest rather than trusting that it exists.

`order_event.order_fee` is the commission the fee model charged. Sum these and compare against Total Fees on the results page.

## 2. What backtests cannot see

Four biases, ordered by how often they wreck a student's first strategy.

**1. Look-ahead bias**: using information that did not exist yet. Module 5 covered the pandas version. In an algorithm it sneaks in through `history` calls that include the current bar, or through fundamental data used before its filing date.

**2. Survivorship bias**: testing on the stocks that exist *today*. A universe of "current S&P 500 members" tested back to 2010 excludes every company that went bankrupt or got delisted, which is exactly the set that would have hurt you. QuantConnect's universe selection is point-in-time and does not have this problem; a CSV of tickers you scraped last week does.

**3. Data-snooping / overfitting**: trying 200 parameter combinations and reporting the best. Section 8 makes this concrete.

**4. Cost blindness**: a strategy that trades daily can look excellent gross and be unprofitable net. Section 6 measures this.

> 🧠 The unifying idea: a backtest measures **the strategy plus your assumptions**. When the result is surprisingly good, suspect the assumptions before believing the strategy.

## Part 2: Analytics

## 3. Return statistics

Start with the equity curve, because every other statistic is derived from it. `(1 + r).cumprod()` is the growth of one dollar, which Module 4 introduced.

In [2]:
equity = (1 + strat_ret).cumprod()
bench_equity = (1 + bench_ret).cumprod()

total_return = equity.iloc[-1] - 1
years = len(strat_ret) / 252
cagr = equity.iloc[-1] ** (1 / years) - 1

print(f"sample length     : {years:.2f} years")
print(f"total return      : {total_return:.2%}")
print(f"CAGR              : {cagr:.2%}")
print(f"benchmark total   : {bench_equity.iloc[-1] - 1:.2%}")

sample length     : 4.00 years
total return      : 56.00%
CAGR              : 11.77%
benchmark total   : 145.93%


**Total return** is what you made. **CAGR** is the constant annual rate that would have produced it, the right number to compare across strategies with different sample lengths.

Now look at the last line. The benchmark made far more than the strategy did. Hold that thought: we will spend the rest of this module discovering that a strategy with entirely respectable risk statistics can still be a worse choice than doing nothing, and that no single number tells you so.

Note we compute `years` as `len(returns) / 252` rather than from calendar dates. Using trading days keeps it consistent with the √252 annualization used everywhere else.

Now the risk side. Module 4 introduced annualized volatility; here it is alongside its more useful cousin.

In [3]:
ann_vol = strat_ret.std() * np.sqrt(252)
ann_ret = strat_ret.mean() * 252
sharpe = ann_ret / ann_vol

# Downside deviation: only moves BELOW zero count as risk.
downside = np.minimum(strat_ret, 0.0)
downside_dev = np.sqrt((downside ** 2).mean()) * np.sqrt(252)
sortino = ann_ret / downside_dev

print(f"annualized vol    : {ann_vol:.2%}")
print(f"downside deviation: {downside_dev:.2%}")
print(f"Sharpe            : {sharpe:.3f}")
print(f"Sortino           : {sortino:.3f}")

annualized vol    : 14.07%
downside deviation: 9.98%
Sharpe            : 0.861
Sortino           : 1.215


**Sharpe** divides annualized return by annualized volatility. It penalises *all* variability, including the upside kind. A strategy that occasionally makes +8% in a day is punished for it.

**Sortino** fixes that by only counting downside moves as risk: square the negative returns, average, take the root. Sortino is always larger than Sharpe for a strategy with positive skew, and the gap tells you something real about the shape of the return distribution.

> A note on the risk-free rate. The textbook Sharpe subtracts it: `(return − rf) / vol`. We use `rf = 0` throughout this curriculum, which is standard for strategy comparison and was the right approximation for most of the 2010s. If you quote Sharpe for a competition or a write-up, say which convention you used.

### ✏️ Your turn: return and risk statistics

From `strat_ret`, compute these five floats:

| variable | definition |
|---|---|
| `my_total` | total compounded return over the sample |
| `my_cagr` | compound annual growth rate, using `len(strat_ret) / 252` years |
| `my_vol` | annualized volatility |
| `my_sharpe` | annualized return (`mean × 252`) ÷ annualized volatility |
| `my_sortino` | annualized return ÷ downside deviation, where downside deviation is `sqrt(mean(min(r, 0)²)) × √252` |

In [4]:
my_total = (1 + strat_ret).prod() - 1
my_cagr = (1 + strat_ret).prod() ** (252 / len(strat_ret)) - 1
my_vol = strat_ret.std() * np.sqrt(252)
my_sharpe = (strat_ret.mean() * 252) / my_vol
my_sortino = (strat_ret.mean() * 252) / (
    np.sqrt((np.minimum(strat_ret, 0.0) ** 2).mean()) * np.sqrt(252))

print(f"total {my_total:.2%} | CAGR {my_cagr:.2%} | vol {my_vol:.2%}")
print(f"Sharpe {my_sharpe:.3f} | Sortino {my_sortino:.3f}")

total 56.00% | CAGR 11.77% | vol 14.07%
Sharpe 0.861 | Sortino 1.215


In [5]:
_tot = float((1 + strat_ret).prod() - 1)
_cagr = float((1 + strat_ret).prod() ** (252 / len(strat_ret)) - 1)
_vol = float(strat_ret.std() * np.sqrt(252))
_shp = float((strat_ret.mean() * 252) / _vol)
_dd = float(np.sqrt((np.minimum(strat_ret, 0.0) ** 2).mean()) * np.sqrt(252))
_srt = float((strat_ret.mean() * 252) / _dd)
assert np.isclose(float(my_total), _tot), "my_total should be (1 + r).prod() - 1"
assert np.isclose(float(my_cagr), _cagr), "my_cagr is off - annualize with 252 / len(r)"
assert np.isclose(float(my_vol), _vol), "my_vol should be std * sqrt(252)"
assert np.isclose(float(my_sharpe), _shp), "my_sharpe should be ann_return / ann_vol"
assert np.isclose(float(my_sortino), _srt), \
    "my_sortino is off - downside deviation uses min(r, 0), squared, averaged, then sqrt"
assert my_sortino > my_sharpe, "Sortino should exceed Sharpe here - check your downside deviation"
print(f"✅ Correct!  Sharpe {_shp:.3f} vs Sortino {_srt:.3f}",
      f"- the gap says the downside is milder than total volatility suggests")

✅ Correct!  Sharpe 0.861 vs Sortino 1.215 - the gap says the downside is milder than total volatility suggests


## 3b. Reading returns by calendar period

A single CAGR hides everything interesting. The first thing a professional does with a backtest is break the returns down by year, because "Sharpe 1.1" and "made all its money in one year" are consistent with each other.

Module 4 introduced `resample` for changing frequency. The same tool works here, with one twist: to compound daily returns into a period return you cannot sum them, you have to use `(1 + r).prod() - 1`.

In [6]:
annual = strat_ret.resample("YE").apply(lambda r: (1 + r).prod() - 1)
annual_bench = bench_ret.resample("YE").apply(lambda r: (1 + r).prod() - 1)

by_year = pd.DataFrame({
    "strategy": annual,
    "benchmark": annual_bench,
    "excess": annual - annual_bench,
})
by_year.index = by_year.index.year
by_year

,strategy,benchmark,excess
2020,-0.0544,0.2534,-0.3079
2021,0.0594,0.3194,-0.2600
2022,0.4509,0.3335,0.1174
2023,0.0733,0.1152,-0.0419


`"YE"` is year-end, the sibling of the `"ME"` (month-end) alias from Module 4. Setting `.index = .index.year` replaces the December-31 timestamps with plain integers, which reads better.

Look at the `excess` column rather than the `strategy` column. A year where you made 12% and the benchmark made 20% is a year you lost, in the only sense that matters to whoever is funding you.

The monthly version is the classic hedge-fund tear-sheet table: months across, years down.

In [7]:
monthly = strat_ret.resample("ME").apply(lambda r: (1 + r).prod() - 1)

table = pd.DataFrame({
    "year": monthly.index.year,
    "month": monthly.index.month,
    "ret": monthly.values,
}).pivot(index="year", columns="month", values="ret")

table.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                 "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][:len(table.columns)]
(table * 100).round(2)

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
year,,,,,,,,,,,,
2020,0.00,0.00,0.00,0.00,0.00,0.00,0.50,2.38,-8.26,0.00,0.00,0.18
2021,-1.37,1.42,-5.28,4.81,-4.75,2.79,-4.21,-2.43,-1.44,0.88,4.36,12.35
2022,6.94,7.26,4.40,7.37,2.70,0.76,1.59,5.41,10.31,-0.70,0.83,-7.80
2023,0.00,0.00,0.00,0.00,0.00,-0.77,-0.99,3.53,-3.87,6.58,3.00,NaN


`pivot` is the reshaping cousin of `unstack` from Module 4: it turns two columns into the row and column axes of a grid, and a third into the values. Here that converts a flat list of 48 monthly returns into a 4×12 grid.

Read it for **clustering**. Losses spread evenly across the table are noise; losses stacked in consecutive cells are a regime the strategy cannot handle, and that is the thing that will happen to you again.

Two reading notes. A `0.00` cell is a month the strategy sat flat and made nothing, not a month it broke even. The trend filter kept it out of the market for the whole first half of 2020 and again in early 2023. And `NaN` means the month falls outside the sample entirely, which is why December 2023 is empty.

The annual table above is already damning: the strategy beat the benchmark in exactly one year out of four, and that one year was the bear market. That is a real and recognisable profile, not a bug. A trend follower's job is to sidestep the bad year, and it pays for that insurance by lagging in the good ones.

## 4. Drawdown: the statistic that decides whether you can actually run the strategy

**Drawdown** is how far below its own previous peak the equity curve is, at every point in time. **Maximum drawdown** is the worst of those.

It matters more than Sharpe for a practical reason: Sharpe describes the average experience, drawdown describes the worst one. A strategy with a 60% drawdown gets switched off by its operator (or its risk committee) long before it recovers, no matter what its Sharpe was.

```
drawdown_t = equity_t / running_max(equity)_t − 1
```

`cummax()` is the running maximum, so this is a two-line calculation. You have computed max drawdown twice already, in NumPy in Module 1, and inside `research_summary` in Module 4. What is new here is everything *around* the maximum: how long it lasted, how often you were under water, and what it does to the return you can claim.

In [8]:
running_max = equity.cummax()
drawdown = equity / running_max - 1

max_dd = drawdown.min()
max_dd_date = drawdown.idxmin()

print(f"max drawdown      : {max_dd:.2%}")
print(f"reached on        : {max_dd_date.date()}")
print(f"peak before it    : {running_max.loc[max_dd_date]:.4f}")
print(f"equity at trough  : {equity.loc[max_dd_date]:.4f}")

max drawdown      : -20.59%
reached on        : 2021-09-09
peak before it    : 1.0666
equity at trough  : 0.8469


**Drawdown duration**, how long you spent below water, is the other half of the story, and it is often the more painful one. A 15% drawdown that recovers in a month is a bad quarter; a 15% drawdown that takes three years to recover is a career.

We measure it as the longest run of consecutive days with `drawdown < 0`.

In [9]:
under_water = drawdown < 0

# Group consecutive True runs: each time the flag flips, start a new group id.
group_id = (under_water != under_water.shift()).cumsum()
run_lengths = under_water.groupby(group_id).sum()      # True counts as 1

longest = int(run_lengths.max())
print(f"longest underwater stretch : {longest} trading days (~{longest / 21:.1f} months)")
print(f"share of days under water  : {under_water.mean():.1%}")

longest underwater stretch : 371 trading days (~17.7 months)
share of days under water  : 78.8%


That `(flag != flag.shift()).cumsum()` idiom is worth memorising. It assigns a new integer id every time the flag changes, which turns "consecutive runs" into an ordinary `groupby`: the split-apply-combine pattern from Module 2, applied to time.

**Calmar ratio** packages return and drawdown into one number: `CAGR / |max drawdown|`. It is the trend-following world's favourite statistic, because it speaks directly to the tolerability of the strategy.

In [10]:
calmar = cagr / abs(max_dd)
print(f"Calmar ratio: {calmar:.3f}  (CAGR {cagr:.2%} / max DD {abs(max_dd):.2%})")

Calmar ratio: 0.572  (CAGR 11.77% / max DD 20.59%)


### ✏️ Your turn: drawdown analysis

From `strat_ret`, compute:

| variable | definition |
|---|---|
| `my_equity` | the equity curve, `(1 + r).cumprod()` |
| `my_dd` | the drawdown series, `equity / equity.cummax() − 1` |
| `my_max_dd` | the worst (most negative) drawdown, a float |
| `my_dd_days` | the **longest** run of consecutive days with `drawdown < 0`, an int |
| `my_calmar` | `CAGR / abs(max drawdown)` |

In [11]:
my_equity = (1 + strat_ret).cumprod()
my_dd = my_equity / my_equity.cummax() - 1

my_max_dd = float(my_dd.min())

uw = my_dd < 0
my_dd_days = int(uw.groupby((uw != uw.shift()).cumsum()).sum().max())

my_calmar = ((1 + strat_ret).prod() ** (252 / len(strat_ret)) - 1) / abs(my_max_dd)

print(f"max DD {my_max_dd:.2%} over {my_dd_days} days | Calmar {my_calmar:.3f}")

max DD -20.59% over 371 days | Calmar 0.572


In [12]:
_eq = (1 + strat_ret).cumprod()
_dd = _eq / _eq.cummax() - 1
_uw = _dd < 0
_days = int(_uw.groupby((_uw != _uw.shift()).cumsum()).sum().max())
_cal = float(((1 + strat_ret).prod() ** (252 / len(strat_ret)) - 1) / abs(_dd.min()))
assert np.allclose(my_equity.values, _eq.values), "my_equity should be (1 + r).cumprod()"
assert np.allclose(my_dd.values, _dd.values), "my_dd should be equity / equity.cummax() - 1"
assert my_dd.max() <= 1e-12, "drawdown should never be positive"
assert np.isclose(float(my_max_dd), float(_dd.min())), "my_max_dd should be the minimum of the series"
assert int(my_dd_days) == _days, (
    f"expected a longest underwater run of {_days} days, got {my_dd_days} - "
    "group consecutive runs with (flag != flag.shift()).cumsum()")
assert np.isclose(float(my_calmar), _cal), "my_calmar should be CAGR / abs(max_dd)"
print(f"✅ Correct!  Worst drawdown {float(_dd.min()):.2%},",
      f"longest underwater stretch {_days} trading days")

✅ Correct!  Worst drawdown -20.59%, longest underwater stretch 371 trading days


## 5. Trade statistics: win rate and profit factor

Return statistics describe the equity curve. Trade statistics describe your *behaviour*, and they are how you diagnose a strategy rather than merely score it.

| Statistic | Definition | Reads as |
|---|---|---|
| **Win rate** | share of trades that made money | how often you are right |
| **Profit factor** | total gains ÷ total losses (absolute) | how much you make when right, relative to losing |
| **Average win / average loss** | mean winner ÷ mean loser | the payoff ratio |
| **Turnover** | how often you trade | how much cost you will pay |

There are **two levels** at which you can measure these, and confusing them is a common reporting error.

The **daily** level asks "on the days I held a position, how often was the day green?". It is easy to compute and it is what most quick pandas backtests report.

In [13]:
active = strat_ret[position.shift(1) == 1.0]      # days we actually held a position

daily_win_rate = (active > 0).sum() / len(active)
daily_profit_factor = active[active > 0].sum() / abs(active[active < 0].sum())

print(f"active days        : {len(active)} of {len(strat_ret)}")
print(f"daily win rate     : {daily_win_rate:.1%}")
print(f"daily profit factor: {daily_profit_factor:.3f}")

active days        : 567 of 1007
daily win rate     : 56.8%
daily profit factor: 1.216


The **trade** level asks "of the round trips I actually took, how many made money?". This is what QuantConnect's results page means by Win Rate, and it is the more meaningful number, because a trade is the thing you decided to do. A day is not.

To get there we need to compound each holding period into a single trade return. That is the run-grouping idiom from the drawdown section again: a new trade begins every time the held/not-held flag flips.

In [14]:
held = position.shift(1).fillna(0.0) == 1.0

# A new group id every time we switch between held and flat.
trade_id = (held != held.shift()).cumsum()

# Compound the daily returns within each held stretch into one trade return.
trades = strat_ret[held].groupby(trade_id[held]).apply(lambda x: (1 + x).prod() - 1)
trades.name = "trade_return"

print(f"round trips: {len(trades)}")
print(f"best  : {trades.max():+.2%}")
print(f"worst : {trades.min():+.2%}")
(trades * 100).round(2).to_frame().T

round trips: 14
best  : +71.61%
worst : -5.61%


close,2,4,6,8,10,12,14,16,18,20,22,24,26,28
trade_return,-5.61,-3.92,-2.05,-2.12,3.05,0.22,-0.58,-1.87,0.91,-4.19,71.61,-2.66,0.94,9.24


In [15]:
wins = trades[trades > 0]
losses = trades[trades < 0]

win_rate = len(wins) / len(trades)
profit_factor = wins.sum() / abs(losses.sum())
payoff = wins.mean() / abs(losses.mean())

print(f"trades        : {len(trades)}  ({len(wins)} winners, {len(losses)} losers)")
print(f"win rate      : {win_rate:.1%}")
print(f"profit factor : {profit_factor:.3f}")
print(f"avg win/loss  : {payoff:.2f}x")

trades        : 14  (6 winners, 8 losers)
win rate      : 42.9%
profit factor : 3.736
avg win/loss  : 4.98x


There is the lesson. This strategy is **wrong more often than it is right**, and it makes money comfortably, because the average winner is several times the average loser.

That is the signature of trend following, and it is why **win rate on its own tells you nothing**. Trend followers typically win 35–45% of their trades and profit from a few large winners. Mean-reversion strategies often win 65%+ of trades and still lose money, because the rare losses are enormous. Always read win rate alongside profit factor and the payoff ratio.

Note also how differently the two levels read: the daily win rate is *above* 50% while the trade win rate is *below* it. Neither is wrong. They answer different questions, so state which one you are quoting.

**Turnover** is what you will pay for. Since `position` is a 0/1 series, `position.diff().abs()` is 1.0 on every day you entered or exited and 0.0 otherwise, so its sum is the number of position changes.

In [16]:
trades = position.diff().abs()
n_changes = int(trades.sum())
annual_turnover = trades.sum() / (len(position) / 252)

print(f"position changes    : {n_changes}")
print(f"round trips (approx): {n_changes // 2}")
print(f"changes per year    : {annual_turnover:.1f}")

position changes    : 27
round trips (approx): 13
changes per year    : 6.8


### ✏️ Your turn: trade statistics

Write `trade_returns(returns, positions)` that compounds each holding period into a single trade return and returns them as a Series, the same construction as above, but as a reusable function.

Then, applying it to `strat_ret` and `position`:

| variable | definition |
|---|---|
| `my_trades` | the Series of round-trip returns |
| `my_win_rate` | share of trades that were **strictly positive** |
| `my_payoff` | mean winner ÷ absolute mean loser |
| `my_changes` | the number of position changes, as an int |

Reminder of the idiom: build the held mask with `positions.shift(1).fillna(0.0) == 1.0`, group with `(held != held.shift()).cumsum()`, and compound each group with `(1 + x).prod() - 1`.

In [17]:
def trade_returns(returns, positions):
    held = positions.shift(1).fillna(0.0) == 1.0
    trade_id = (held != held.shift()).cumsum()
    return returns[held].groupby(trade_id[held]).apply(lambda x: (1 + x).prod() - 1)


my_trades = trade_returns(strat_ret, position)

my_win_rate = (my_trades > 0).sum() / len(my_trades)
my_payoff = my_trades[my_trades > 0].mean() / abs(my_trades[my_trades < 0].mean())

my_changes = int(position.diff().abs().sum())

print(f"{len(my_trades)} trades | win rate {my_win_rate:.1%} "
      f"| payoff {my_payoff:.2f}x | {my_changes} changes")

14 trades | win rate 42.9% | payoff 4.98x | 27 changes


In [18]:
_held = position.shift(1).fillna(0.0) == 1.0
_tr = strat_ret[_held].groupby((_held != _held.shift()).cumsum()[_held]).apply(
    lambda x: (1 + x).prod() - 1)
_wr = float((_tr > 0).sum() / len(_tr))
_po = float(_tr[_tr > 0].mean() / abs(_tr[_tr < 0].mean()))
_ch = int(position.diff().abs().sum())
assert len(my_trades) == len(_tr), (
    f"expected {len(_tr)} round trips, got {len(my_trades)} - each unbroken stretch of "
    "held days is ONE trade")
assert np.allclose(np.sort(my_trades.values), np.sort(_tr.values)), \
    "the trade returns are off - compound within each stretch using (1 + x).prod() - 1"
assert np.isclose(float(my_win_rate), _wr), "my_win_rate should be the share of positive trades"
assert np.isclose(float(my_payoff), _po), \
    "my_payoff should be mean winner / abs(mean loser)"
assert int(my_changes) == _ch, f"expected {_ch} position changes, got {my_changes}"
assert my_win_rate < 0.5 and my_payoff > 1, "sanity: this strategy wins rarely but wins big"
print(f"✅ Correct!  {len(_tr)} trades, win rate {_wr:.1%}, payoff {_po:.2f}x",
      "- being right less than half the time is fine when the winners are this much bigger")

✅ Correct!  14 trades, win rate 42.9%, payoff 4.98x - being right less than half the time is fine when the winners are this much bigger


## 6. Costs: the statistic that kills most student strategies

Every position change costs money: commission, exchange fees, the bid-ask spread, and market impact. Together these are usually quoted in **basis points** (1 bp = 0.01%).

A realistic all-in round-trip cost for liquid US equities is roughly **5–10 bp**. That sounds negligible. Applied to a strategy that changes position 200 times a year, it is 1–2% of annual return, which is often the entire edge.

Model it as: on any day the position changes, subtract `|change| × cost`.

In [19]:
def apply_costs(returns, positions, cost_bps):
    turnover = positions.diff().abs().fillna(0.0)
    cost = turnover * (cost_bps / 10_000)
    return returns - cost


for bps in [0, 2, 5, 10, 25]:
    net = apply_costs(strat_ret, position, bps)
    total = (1 + net).prod() - 1
    shp = (net.mean() * 252) / (net.std() * np.sqrt(252))
    print(f"  {bps:>2} bp per change -> total {total:>8.2%}   Sharpe {shp:.3f}")

   0 bp per change -> total   56.00%   Sharpe 0.861
   2 bp per change -> total   55.15%   Sharpe 0.851
   5 bp per change -> total   53.90%   Sharpe 0.837
  10 bp per change -> total   51.82%   Sharpe 0.812
  25 bp per change -> total   45.76%   Sharpe 0.738


Read the ladder. This strategy changes position rarely, so it survives; a daily-rebalancing version of the same idea would be wiped out by the same cost assumptions.

> 🧠 **The rule.** Before you get attached to a backtest, rerun it at 10 bp and 25 bp. If the edge disappears, you do not have a strategy, you have a cost-free fantasy. QuantConnect applies real fees when you set a brokerage model, which is why section 1 told you to set one.

Cost sensitivity is also a good *design* signal. If halving your turnover barely changes gross returns but doubles net returns, trade less.

### ✏️ Your turn: cost sensitivity

Write `net_sharpe(returns, positions, cost_bps)` returning the **annualized Sharpe ratio** after costs, where cost on each day is `|position change| × cost_bps / 10000`.

Then set:
- `sharpe_0`: the Sharpe at 0 bp
- `sharpe_10`: the Sharpe at 10 bp
- `breakeven_bps`: the **smallest whole number** of basis points, searching `0, 1, 2, … 200`, at which the net Sharpe first drops below `0.5`. Use `None` if it never does.

In [20]:
def net_sharpe(returns, positions, cost_bps):
    cost = positions.diff().abs().fillna(0.0) * (cost_bps / 10_000)
    net = returns - cost
    return (net.mean() * 252) / (net.std() * np.sqrt(252))


sharpe_0 = net_sharpe(strat_ret, position, 0)
sharpe_10 = net_sharpe(strat_ret, position, 10)

breakeven_bps = None
for bps in range(0, 201):
    if net_sharpe(strat_ret, position, bps) < 0.5:
        breakeven_bps = bps
        break

print(f"0 bp: {sharpe_0:.3f} | 10 bp: {sharpe_10:.3f} | drops below 0.5 at {breakeven_bps} bp")

0 bp: 0.861 | 10 bp: 0.812 | drops below 0.5 at 74 bp


In [21]:
def _ns(returns, positions, bps):
    cost = positions.diff().abs().fillna(0.0) * (bps / 10_000)
    net = returns - cost
    return float((net.mean() * 252) / (net.std() * np.sqrt(252)))

_be = None
for _b in range(0, 201):
    if _ns(strat_ret, position, _b) < 0.5:
        _be = _b
        break
assert np.isclose(float(net_sharpe(strat_ret, position, 0)), _ns(strat_ret, position, 0)), \
    "net_sharpe(…, 0) should equal the gross Sharpe"
assert np.isclose(float(net_sharpe(strat_ret, position, 7)), _ns(strat_ret, position, 7)), \
    "net_sharpe is off at 7 bp - cost is |position change| * bps / 10000"
assert np.isclose(float(sharpe_0), _ns(strat_ret, position, 0)), "sharpe_0 is off"
assert np.isclose(float(sharpe_10), _ns(strat_ret, position, 10)), "sharpe_10 is off"
assert sharpe_10 < sharpe_0, "costs should reduce the Sharpe ratio"
assert breakeven_bps == _be, f"expected breakeven_bps to be {_be}, got {breakeven_bps}"
print(f"✅ Correct!  Sharpe falls from {_ns(strat_ret, position, 0):.3f} to",
      f"{_ns(strat_ret, position, 10):.3f} at 10 bp; below 0.5 at {_be} bp")

✅ Correct!  Sharpe falls from 0.861 to 0.812 at 10 bp; below 0.5 at 74 bp


## 7. Relative performance: beta and alpha

QuantConnect reports your strategy against a benchmark (SPY by default, or whatever you passed to `set_benchmark` in Module 3). Two numbers summarise the relationship.

**Beta** is your sensitivity to the benchmark: `cov(strategy, benchmark) / var(benchmark)`. A beta of 0.5 means that when the market moves 1%, you tend to move 0.5%.

**Alpha** is what is left over once beta is accounted for: `strategy_annual_return − beta × benchmark_annual_return`. It is the part of your return that the market did not hand you.

The distinction matters because a long-only trend follower on SPY is *mostly* just holding SPY part of the time. Its return is not evidence of skill until you have removed the beta.

In [22]:
cov = np.cov(strat_ret, bench_ret)[0, 1]
var_b = np.var(bench_ret, ddof=1)
beta = cov / var_b

ann_strat = strat_ret.mean() * 252
ann_bench = bench_ret.mean() * 252
alpha = ann_strat - beta * ann_bench

print(f"beta            : {beta:.3f}")
print(f"strategy return : {ann_strat:.2%} annualized")
print(f"benchmark return: {ann_bench:.2%} annualized")
print(f"alpha           : {alpha:.2%} annualized")

beta            : 0.551
strategy return : 12.12% annualized
benchmark return: 24.32% annualized
alpha           : -1.28% annualized


`np.cov(a, b)` returns the 2×2 covariance matrix, so `[0, 1]` picks the off-diagonal covariance term. `ddof=1` gives the sample variance, matching `np.cov`'s default. Mixing the two conventions is a common source of small, confusing discrepancies.

A beta below 1 is expected here: the strategy is flat much of the time, so it simply cannot track the benchmark one-for-one.

The alpha is the verdict, and it is **negative**. Every bit of this strategy's return, and then some, is explained by the market exposure it happened to be carrying. Holding the benchmark at a constant 55% weight, with no signal, no crossover and no work, would have done slightly better.

That is worth sitting with, because the Sharpe of 0.861 and the Sortino of 1.215 above are perfectly respectable numbers. They are not wrong. They simply do not answer the question "was this worth doing?"

### Tracking error and the information ratio

Beta and alpha decompose your return. The **information ratio** scores it. Define the *active return* as your return minus the benchmark's, day by day:

- **Tracking error** = annualized volatility of the active return. How far you drift from the benchmark.
- **Information ratio** = annualized active return ÷ tracking error.

It is Sharpe measured in relative terms, and it is the right statistic whenever your job is to beat a benchmark rather than to make money in absolute terms. An information ratio above 0.5 is respectable; above 1.0 is excellent and rare.

In [23]:
active_ret = strat_ret - bench_ret

tracking_error = active_ret.std() * np.sqrt(252)
information_ratio = (active_ret.mean() * 252) / tracking_error

print(f"active return (ann) : {active_ret.mean() * 252:.2%}")
print(f"tracking error      : {tracking_error:.2%}")
print(f"information ratio   : {information_ratio:.3f}")
print(f"(for comparison) Sharpe: {sharpe:.3f}")

active return (ann) : -12.20%
tracking error      : 12.70%
information ratio   : -0.960
(for comparison) Sharpe: 0.861


The two ratios disagree completely, and that is the point of showing them together. Sharpe asks *"was this worth the risk?"* and answers yes. The information ratio asks *"was this worth doing instead of just buying the benchmark?"* and answers, emphatically, no.

Both are correct. They are different questions, and which one you should care about depends entirely on what your alternative was. If your alternative was cash, the Sharpe is the relevant number. If your alternative was buying SPY and going to lunch (which, for a student running a strategy on SPY, it was) the information ratio is the relevant number.

> 🧠 **Report both.** A write-up that quotes only the flattering one is not analysis, it is advertising. The habit that makes you trustworthy is volunteering the statistic that undercuts your own result.

### ✏️ Your turn: tracking error and information ratio

From `strat_ret` and `bench_ret` compute:

| variable | definition |
|---|---|
| `my_active` | the daily active return, strategy minus benchmark |
| `my_te` | tracking error: `my_active.std() × √252` |
| `my_ir` | information ratio: `(my_active.mean() × 252) / my_te` |
| `beats_benchmark` | a **bool**: is the annualized active return positive? |

In [24]:
my_active = strat_ret - bench_ret
my_te = my_active.std() * np.sqrt(252)
my_ir = (my_active.mean() * 252) / my_te

beats_benchmark = bool(my_active.mean() * 252 > 0)

print(f"tracking error {my_te:.2%} | information ratio {my_ir:.3f} "
      f"| beats benchmark: {beats_benchmark}")

tracking error 12.70% | information ratio -0.960 | beats benchmark: False


In [25]:
_act = strat_ret - bench_ret
_te = float(_act.std() * np.sqrt(252))
_ir = float((_act.mean() * 252) / _te)
assert np.allclose(my_active.values, _act.values), \
    "my_active should be the daily difference, strategy minus benchmark"
assert np.isclose(float(my_te), _te), "my_te should be the active return's std times sqrt(252)"
assert np.isclose(float(my_ir), _ir), \
    "my_ir should be annualized active return divided by tracking error"
assert isinstance(beats_benchmark, (bool, np.bool_)), "beats_benchmark should be a bool"
assert bool(beats_benchmark) == bool(_act.mean() * 252 > 0), \
    "beats_benchmark should test whether the annualized active return is positive"
print(f"✅ Correct!  Tracking error {_te:.2%}, information ratio {_ir:.3f}")

✅ Correct!  Tracking error 12.70%, information ratio -0.960


### ✏️ Your turn: beta and alpha

From `strat_ret` and `bench_ret` compute:

| variable | definition |
|---|---|
| `my_beta` | `cov(strategy, benchmark) / var(benchmark)`, using `np.cov` and `ddof=1` |
| `my_alpha` | annualized strategy return − `my_beta` × annualized benchmark return |
| `my_corr` | the plain correlation between the two return series |

All three should be floats.

In [26]:
my_beta = np.cov(strat_ret, bench_ret)[0, 1] / np.var(bench_ret, ddof=1)

my_alpha = strat_ret.mean() * 252 - my_beta * bench_ret.mean() * 252

my_corr = strat_ret.corr(bench_ret)

print(f"beta {my_beta:.3f} | alpha {my_alpha:.2%} | corr {my_corr:.3f}")

beta 0.551 | alpha -1.28% | corr 0.741


In [27]:
_b = float(np.cov(strat_ret, bench_ret)[0, 1] / np.var(bench_ret, ddof=1))
_a = float(strat_ret.mean() * 252 - _b * bench_ret.mean() * 252)
_c = float(strat_ret.corr(bench_ret))
assert np.isclose(float(my_beta), _b), "my_beta should be cov / var of the benchmark"
assert np.isclose(float(my_alpha), _a), \
    "my_alpha should be annualized strategy return minus beta times annualized benchmark return"
assert np.isclose(float(my_corr), _c), "my_corr should be the correlation of the two return series"
assert 0 < my_beta < 1, "a partly-invested long-only strategy should have beta between 0 and 1"
print(f"✅ Correct!  Beta {_b:.3f}, alpha {_a:.2%} annualized")

✅ Correct!  Beta 0.551, alpha -1.28% annualized


## 8. How to not fool yourself

This is the part that separates a research process from a slot machine.

### 8.1 In-sample and out-of-sample

Split your data. Build the strategy on the first chunk, and *then* look at the second. If the second chunk disagrees, believe the second chunk. It is the only part you did not train on.

The standard split is 70/30 or 2/3–1/3, chronologically. Never shuffle time-series data: randomly splitting days lets the model learn from the future of its own test set.

In [28]:
split = int(len(strat_ret) * 0.7)
in_sample = strat_ret.iloc[:split]
out_sample = strat_ret.iloc[split:]


def sharpe_of(r):
    return (r.mean() * 252) / (r.std() * np.sqrt(252))


comparison = pd.DataFrame({
    "days": [len(in_sample), len(out_sample)],
    "total_return": [float((1 + in_sample).prod() - 1), float((1 + out_sample).prod() - 1)],
    "sharpe": [float(sharpe_of(in_sample)), float(sharpe_of(out_sample))],
}, index=["in-sample (first 70%)", "out-of-sample (last 30%)"])
comparison

,days,total_return,sharpe
in-sample (first 70%),704,0.5987,1.3198
out-of-sample (last 30%),303,-0.0242,-0.0540


### 8.2 Was it one lucky year?

A single Sharpe over four years averages away the answer to the question you most want answered: was the edge *there the whole time*, or is one exceptional stretch carrying the number?

The rolling Sharpe answers it directly. Module 5 built rolling windows for indicators; the same `.rolling()` applies to returns.

In [29]:
window = 252      # one trading year

roll_mean = strat_ret.rolling(window).mean() * 252
roll_vol = strat_ret.rolling(window).std() * np.sqrt(252)
rolling_sharpe = (roll_mean / roll_vol).dropna()

print(f"rolling 1-year Sharpe over {len(rolling_sharpe)} windows")
print(f"  best   : {rolling_sharpe.max():.2f}  ({rolling_sharpe.idxmax().date()})")
print(f"  median : {rolling_sharpe.median():.2f}")
print(f"  worst  : {rolling_sharpe.min():.2f}  ({rolling_sharpe.idxmin().date()})")
print(f"  share of windows above 0: {(rolling_sharpe > 0).mean():.1%}")
rolling_sharpe.resample("QE").last().round(2)

rolling 1-year Sharpe over 756 windows
  best   : 4.56  (2022-09-15)
  median : 0.50
  worst  : -1.07  (2021-08-05)
  share of windows above 0: 59.3%


2020-12-31   -0.51
2021-03-31   -0.73
2021-06-30   -0.44
2021-09-30   -0.68
2021-12-31    0.79
2022-03-31    2.81
2022-06-30    3.75
2022-09-30    4.08
2022-12-31    1.84
2023-03-31    0.98
2023-06-30    0.40
2023-09-30   -0.46
2023-12-31   -0.63
Freq: QE-DEC, Name: close, dtype: float64

The spread between best and worst is the honest picture. Our strategy's rolling Sharpe swings from strongly positive to solidly negative depending on which year you ask about. It does not really "have a Sharpe of 0.86"; it has several different behaviours, and the headline number is their average.

The number to quote in a write-up is the **median** rolling Sharpe, not the full-sample one. And `(rolling_sharpe > 0).mean()` (the share of years you would have made money) is often more persuasive to a sceptical reader than any single ratio.

### 8.3 Parameter stability

A robust strategy works across a *neighbourhood* of parameters. If 20/60 is excellent and 19/60 and 21/60 are terrible, you have found a quirk of this particular price path, not a market effect.

The test: sweep the parameters and look at the whole surface, not the maximum.

In [30]:
def run_params(fast_n, slow_n, trend_n=150):
    f = price.rolling(fast_n).mean()
    s = price.rolling(slow_n).mean()
    t = price.rolling(trend_n).mean()
    pos = ((f > s) & (price > t)).astype(float)
    r = (pos.shift(1) * price.pct_change()).dropna()
    return float((r.mean() * 252) / (r.std() * np.sqrt(252)))


grid = pd.DataFrame(
    [[round(run_params(f, s), 3) for s in [40, 60, 80, 100]] for f in [10, 15, 20, 25]],
    index=[f"fast={f}" for f in [10, 15, 20, 25]],
    columns=[f"slow={s}" for s in [40, 60, 80, 100]],
)
grid

,slow=40,slow=60,slow=80,slow=100
fast=10,0.546,0.875,0.996,1.058
fast=15,0.584,0.810,0.865,0.954
fast=20,0.579,0.861,1.027,0.911
fast=25,0.765,0.832,1.070,0.903


Read the grid as a surface. You want a broad plateau of decent numbers, not one spectacular cell surrounded by bad ones. Report the **median** of the surface alongside the best cell. The median is much closer to what you should expect going forward.

### 8.4 The multiple-testing problem

If you test 100 random strategies on the same data, roughly 5 will look significant at the 5% level *by construction*. This is not a subtle statistical point; it is the main reason published backtests do not replicate.

Rather than assert it, let us measure it. Below we build **200 candidate assets that are pure random walks with zero drift** (a set of markets in which no strategy can possibly have an edge, by construction), and then run one fixed 20/60 crossover across all of them and keep the best. This is exactly what "I scanned the S&P 500 for tickers where my strategy works" does.

In [31]:
snoop_rng = np.random.default_rng(101)


def noise_market(n=1008, vol=0.012):
    # A price series with NO edge: zero drift, independent daily moves.
    return pd.Series(100 * np.exp(np.cumsum(snoop_rng.normal(0, vol, n))), index=dates[:n])


def run_rule(series, fast_n=20, slow_n=60):
    # Sharpe of one fixed shifted crossover, applied to any price series.
    pos = (series.rolling(fast_n).mean() > series.rolling(slow_n).mean()).astype(float)
    r = (pos.shift(1) * series.pct_change()).dropna()
    return float((r.mean() * 252) / (r.std() * np.sqrt(252)))


candidates = [noise_market() for _ in range(200)]
snoop = pd.DataFrame({
    "asset": [f"NOISE_{i:03d}" for i in range(200)],
    "sharpe": [run_rule(c) for c in candidates],
})

print("true edge in every one of these markets: exactly zero, by construction")
print(f"best Sharpe found  : {snoop['sharpe'].max():.3f}")
print(f"median Sharpe      : {snoop['sharpe'].median():.3f}")
print(f"assets above 0.5   : {(snoop['sharpe'] > 0.5).sum()} of 200")
print(f"assets above 1.0   : {(snoop['sharpe'] > 1.0).sum()} of 200")
snoop.sort_values("sharpe", ascending=False).head(5)

true edge in every one of these markets: exactly zero, by construction
best Sharpe found  : 1.246
median Sharpe      : -0.031
assets above 0.5   : 32 of 200
assets above 1.0   : 4 of 200


,asset,sharpe
7,NOISE_007,1.2464
155,NOISE_155,1.1866
150,NOISE_150,1.1322
130,NOISE_130,1.0598
23,NOISE_023,0.9427


Read that carefully. There is **no edge anywhere in this data**, and the search still produced a headline Sharpe that would look publishable, plus dozens of assets clearing 0.5. Nothing went wrong and no bug is hiding: searching 200 candidates and reporting the maximum is a procedure that manufactures impressive numbers out of pure noise.

The **median** tells the truth. It sits near zero, which is the correct answer.

Now the test that catches it. Take the winners and run them on **fresh, independent** data, an honest out-of-sample check.

In [32]:
fresh = [noise_market() for _ in range(200)]      # a new, independent period

ranked = snoop.sort_values("sharpe", ascending=False)
best_i = int(ranked.index[0])

print(f"winner: {ranked.iloc[0]['asset']}")
print(f"  in-sample Sharpe : {ranked.iloc[0]['sharpe']:.3f}")
print(f"  on fresh data    : {run_rule(fresh[best_i]):.3f}")
print()
print(f"top 10 in-sample mean    : {ranked['sharpe'].head(10).mean():.3f}")
print(f"the same 10, fresh data  : "
      f"{np.mean([run_rule(fresh[i]) for i in ranked.index[:10]]):.3f}")

winner: NOISE_007
  in-sample Sharpe : 1.246


  on fresh data    : 0.296

top 10 in-sample mean    : 0.997
the same 10, fresh data  : 0.025


The edge evaporates, because there was never an edge, only a search.

> 🧠 **The lesson.** The gap between "best of 200" and "median of 200" is not skill, it is the size of your search. Report both, and treat the median as your expectation.

Practical defences:

- **Count your attempts.** Keep a log of every variant you tried. A Sharpe of 1.2 found on attempt 3 is a very different claim than the same Sharpe found on attempt 300.
- **Demand an economic story.** "Momentum persists because institutions rebalance slowly" is a hypothesis you can test. "The 37-day MA works" is a number you found.
- **Reserve a holdout you touch once.** If you keep testing against your out-of-sample set, it stops being out-of-sample.

### ✏️ Your turn: measuring your own search

Using the `snoop` DataFrame, the `fresh` list of independent price series, and the `run_rule` helper above:

1. `n_above_one`: how many of the 200 zero-edge assets reached an in-sample Sharpe above **1.0**, as an int.
2. `snoop_gap`: `best Sharpe − median Sharpe` across the 200.
3. `top20_oos`: take the **top 20** by in-sample Sharpe and measure each on its matching `fresh[i]` series; store the **mean** of those 20 out-of-sample Sharpes.
4. `survived`, a **bool**: is `top20_oos` still above 0.5?

The row positions in `snoop` line up with the positions in `fresh`, so `fresh[i]` is the fresh data for the asset in row `i`.

You already know the true answer is zero, so treat this as calibrating your eye for how big a fake number a search can produce.

In [33]:
n_above_one = int((snoop["sharpe"] > 1.0).sum())

snoop_gap = float(snoop["sharpe"].max() - snoop["sharpe"].median())

top20_idx = snoop.sort_values("sharpe", ascending=False).index[:20]
top20_oos = float(np.mean([run_rule(fresh[int(i)]) for i in top20_idx]))

survived = bool(top20_oos > 0.5)

print(f"{n_above_one} of 200 beat Sharpe 1.0 on pure noise | gap {snoop_gap:.3f}")
print(f"top-20 out-of-sample mean {top20_oos:.3f} | survived: {survived}")

4 of 200 beat Sharpe 1.0 on pure noise | gap 1.278
top-20 out-of-sample mean 0.080 | survived: False


In [34]:
_n = int((snoop["sharpe"] > 1.0).sum())
_gap = float(snoop["sharpe"].max() - snoop["sharpe"].median())
_idx = snoop.sort_values("sharpe", ascending=False).index[:20]
_oos = float(np.mean([run_rule(fresh[int(i)]) for i in _idx]))
assert int(n_above_one) == _n, f"expected {_n} assets above Sharpe 1.0, got {n_above_one}"
assert np.isclose(float(snoop_gap), _gap), "snoop_gap should be max minus median"
assert np.isclose(float(top20_oos), _oos), (
    "top20_oos should be the MEAN out-of-sample Sharpe of the 20 best in-sample assets - "
    "make sure fresh[i] uses the same row position i as snoop")
assert isinstance(survived, (bool, np.bool_)), "survived should be a bool"
assert bool(survived) == bool(_oos > 0.5), "survived should test top20_oos > 0.5"
assert not survived, "on zero-edge data the in-sample winners should NOT survive"
print(f"✅ Correct!  {_n}/200 assets cleared Sharpe 1.0 on data with no edge whatsoever;",
      f"the top 20 average {_oos:.3f} out of sample. That gap is the cost of searching.")

✅ Correct!  4/200 assets cleared Sharpe 1.0 on data with no edge whatsoever; the top 20 average 0.080 out of sample. That gap is the cost of searching.


### 8.5 Enough trades to mean anything

The standard error of a Sharpe estimate is roughly `sqrt((1 + Sharpe²/2) / n_years)`. With one year of data, a measured Sharpe of 1.0 has a standard error near 1.0. The true value could easily be 0.

Rules of thumb: **at least 3 years**, preferably 10; **at least 30 round trips**, preferably 100+. A strategy with 6 trades has no statistics, only anecdotes.

Apply that honestly to our own example. It has four years and **14 round trips**, under half the minimum. Everything computed in this notebook is arithmetically correct and statistically thin, and the right way to report it is to say so.

In [35]:
def sharpe_stderr(sharpe, n_years):
    return np.sqrt((1 + sharpe ** 2 / 2) / n_years)


measured = sharpe_of(strat_ret)
for yrs in [1, 2, 4, 10, 20]:
    se = sharpe_stderr(measured, yrs)
    print(f"  {yrs:>2} years of data -> Sharpe {measured:.2f} +/- {se:.2f}"
          f"   (95% CI: {measured - 1.96 * se:>5.2f} to {measured + 1.96 * se:.2f})")

   1 years of data -> Sharpe 0.86 +/- 1.17   (95% CI: -1.43 to 3.16)
   2 years of data -> Sharpe 0.86 +/- 0.83   (95% CI: -0.76 to 2.48)
   4 years of data -> Sharpe 0.86 +/- 0.59   (95% CI: -0.29 to 2.01)
  10 years of data -> Sharpe 0.86 +/- 0.37   (95% CI:  0.14 to 1.59)
  20 years of data -> Sharpe 0.86 +/- 0.26   (95% CI:  0.35 to 1.37)


Look at the one-year row. The confidence interval comfortably contains zero. That is the honest state of knowledge after a one-year backtest, however good the headline number looked.

### ✏️ Your turn: split-sample comparison

Split `strat_ret` chronologically at **60%**:

1. `is_ret`: the first 60% of returns; `oos_ret`: the remaining 40%.
2. `is_sharpe`, `oos_sharpe`: the annualized Sharpe of each.
3. `degradation`: `is_sharpe − oos_sharpe`.

Use integer slicing (`int(len(strat_ret) * 0.6)`) so the split is exact and chronological.

In [36]:
cut = int(len(strat_ret) * 0.6)
is_ret = strat_ret.iloc[:cut]
oos_ret = strat_ret.iloc[cut:]

is_sharpe = (is_ret.mean() * 252) / (is_ret.std() * np.sqrt(252))
oos_sharpe = (oos_ret.mean() * 252) / (oos_ret.std() * np.sqrt(252))
degradation = is_sharpe - oos_sharpe

print(f"in-sample {is_sharpe:.3f} | out-of-sample {oos_sharpe:.3f} "
      f"| degradation {degradation:+.3f}")

in-sample 0.915 | out-of-sample 0.818 | degradation +0.098


In [37]:
_cut = int(len(strat_ret) * 0.6)
_is, _oos = strat_ret.iloc[:_cut], strat_ret.iloc[_cut:]
_iss = float((_is.mean() * 252) / (_is.std() * np.sqrt(252)))
_oss = float((_oos.mean() * 252) / (_oos.std() * np.sqrt(252)))
assert len(is_ret) == _cut and len(oos_ret) == len(strat_ret) - _cut, \
    f"expected a {_cut}/{len(strat_ret) - _cut} split"
assert is_ret.index[0] == strat_ret.index[0], "the in-sample block must come FIRST chronologically"
assert oos_ret.index[-1] == strat_ret.index[-1], "the out-of-sample block must be the LAST 40%"
assert np.isclose(float(is_sharpe), _iss), "is_sharpe is off"
assert np.isclose(float(oos_sharpe), _oss), "oos_sharpe is off"
assert np.isclose(float(degradation), _iss - _oss), "degradation should be is_sharpe - oos_sharpe"
print(f"✅ Correct!  In-sample {_iss:.3f} vs out-of-sample {_oss:.3f}",
      f"({_iss - _oss:+.3f}) - always trust the second number more")

✅ Correct!  In-sample 0.915 vs out-of-sample 0.818 (+0.098) - always trust the second number more


## 9. Reading the QuantConnect results page

When a backtest finishes, QuantConnect shows a statistics table. You have now computed almost all of it by hand. Here is the translation.

| QC statistic | What you computed | Watch out for |
|---|---|---|
| Total Return | `(1 + r).prod() - 1` | meaningless without the sample length |
| Compounding Annual Return | CAGR | the number to compare across strategies |
| Sharpe Ratio | `ann_ret / ann_vol` | QC uses a non-zero risk-free rate; yours may differ slightly |
| Sortino Ratio | `ann_ret / downside_dev` | always ≥ Sharpe for positively skewed returns |
| Probabilistic Sharpe Ratio | — | probability the true Sharpe exceeds 0; low PSR = short sample |
| Drawdown | `min(equity / cummax − 1)` | the number that decides if it is runnable |
| Win Rate / Loss Rate | share of winning trades | meaningless without profit factor |
| Profit-Loss Ratio | average win ÷ average loss | the other half of win rate |
| Alpha / Beta | regression vs benchmark | low beta is how you show it is not just SPY |
| Annual Standard Deviation | `std × √252` | |
| Information Ratio | active return ÷ tracking error | alpha per unit of *relative* risk |
| Treynor Ratio | excess return ÷ beta | rarely decisive |
| Total Fees | sum of commissions | compare against total return |
| Estimated Strategy Capacity | — | how much money the strategy could run |
| Portfolio Turnover | traded value ÷ portfolio value | pairs with Total Fees |

Two on that list you have not computed: **Probabilistic Sharpe Ratio**, which formalises section 8.5 into a single probability, and **Estimated Strategy Capacity**, QuantConnect's estimate of how much capital the strategy could absorb before its own market impact ruins it.

### Plotting your own diagnostics

`self.plot(chart, series, value)` adds a custom chart to the results page. Plotting the things your strategy *thinks* is the fastest way to debug it.

In [ ]:
# 🔵 QC cell, custom charts for debugging
class ChartedStrategy(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2020, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)
        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)

        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.fast = self.sma(self.symbol, 20, Resolution.DAILY)
        self.slow = self.sma(self.symbol, 60, Resolution.DAILY)
        self.set_warm_up(70, Resolution.DAILY)

        # One chart per question you want answered about the strategy.
        self.schedule.on(self.date_rules.every_day(self.symbol),
                         self.time_rules.before_market_close(self.symbol, 5),
                         self.record_state)

    def record_state(self):
        if self.is_warming_up or not self.slow.is_ready:
            return

        self.plot("Signal", "fast", self.fast.current.value)
        self.plot("Signal", "slow", self.slow.current.value)
        self.plot("Exposure", "invested",
                  1 if self.portfolio[self.symbol].invested else 0)
        self.plot("Exposure", "leverage",
                  self.portfolio.total_holdings_value / self.portfolio.total_portfolio_value)

    def on_data(self, data: Slice):
        if self.is_warming_up or not self.slow.is_ready:
            return
        if not data.bars.contains_key(self.symbol):
            return

        long_now = self.fast.current.value > self.slow.current.value
        invested = self.portfolio[self.symbol].invested
        if long_now and not invested:
            self.set_holdings(self.symbol, 1.0)
        elif not long_now and invested:
            self.liquidate(self.symbol)

The `schedule.on(...)` call is Module 3's Pillar 5 doing diagnostic work: run `record_state` a few minutes before each close, so every chart point is sampled at a consistent time of day.

`self.portfolio.total_holdings_value / self.portfolio.total_portfolio_value` is your realised leverage. Plotting it catches the `set_holdings` accident from section 1 immediately. If that line ever goes above 1.0 when you did not intend leverage, you have a sizing bug.

### ✏️ Your turn: write the backtest configuration

Write the `initialize` for a strategy that is set up to be judged honestly:

- **2018-01-01** to **2024-01-01**, **$100,000**
- Interactive Brokers brokerage model, **margin** account
- **SPY** daily, symbol on `self.symbol`
- benchmark set to **SPY**
- a 200-bar SMA on `self.trend`, warmed up with enough bars

This is a 🔵 QC cell. Compare it against the answer key.

In [ ]:
# 🔵 QC cell, no local self-check for this one
class HonestSetup(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2018, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)

        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)

        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.set_benchmark("SPY")

        self.trend = self.sma(self.symbol, 200, Resolution.DAILY)
        self.set_warm_up(210, Resolution.DAILY)

## 🏁 Mini-challenge: the performance report

One function, used for the rest of the curriculum. Every strategy you build in Modules 7–9 gets passed through this.

### ✏️ Your turn: a performance report function

Write `performance_report(returns, benchmark=None, cost_bps=0, positions=None)` returning a **pandas Series** with exactly these entries, in this order:

| key | definition |
|---|---|
| `total_return` | `(1 + r).prod() − 1` |
| `cagr` | `(1 + r).prod() ** (252 / n) − 1` |
| `ann_vol` | `std × √252` |
| `sharpe` | `(mean × 252) / ann_vol` |
| `sortino` | `(mean × 252) / (sqrt(mean(min(r,0)²)) × √252)` |
| `max_drawdown` | `min(equity / equity.cummax() − 1)` |
| `calmar` | `cagr / abs(max_drawdown)` |
| `win_rate` | share of **all** days strictly positive |

If `positions` and a non-zero `cost_bps` are given, subtract `|positions.diff()| × cost_bps / 10000` from the returns **before** computing anything. If `benchmark` is given, append a `beta` entry (`cov / var`, `ddof=1`); otherwise do not include that key at all.

In [38]:
def performance_report(returns, benchmark=None, cost_bps=0, positions=None):
    r = returns
    if positions is not None and cost_bps:
        r = returns - positions.diff().abs().fillna(0.0) * (cost_bps / 10_000)

    equity = (1 + r).cumprod()
    dd = equity / equity.cummax() - 1
    cagr = float((1 + r).prod() ** (252 / len(r)) - 1)

    ann_vol = float(r.std() * np.sqrt(252))
    downside = float(np.sqrt((np.minimum(r, 0.0) ** 2).mean()) * np.sqrt(252))

    out = {
        "total_return": float((1 + r).prod() - 1),
        "cagr": cagr,
        "ann_vol": ann_vol,
        "sharpe": float((r.mean() * 252) / ann_vol),
        "sortino": float((r.mean() * 252) / downside),
        "max_drawdown": float(dd.min()),
        "calmar": cagr / abs(float(dd.min())),
        "win_rate": float((r > 0).mean()),
    }

    if benchmark is not None:
        out["beta"] = float(np.cov(r, benchmark)[0, 1] / np.var(benchmark, ddof=1))

    return pd.Series(out)


performance_report(strat_ret, benchmark=bench_ret, cost_bps=5, positions=position)

total_return    0.5390
cagr            0.1139
ann_vol         0.1409
sharpe          0.8366
sortino         1.1785
max_drawdown   -0.2139
calmar          0.5326
win_rate        0.3198
beta            0.5513
dtype: float64

In [39]:
KEYS = ["total_return", "cagr", "ann_vol", "sharpe", "sortino",
        "max_drawdown", "calmar", "win_rate"]


def _ref(returns, benchmark=None, cost_bps=0, positions=None):
    r = returns
    if positions is not None and cost_bps:
        r = returns - positions.diff().abs().fillna(0.0) * (cost_bps / 10_000)
    eq = (1 + r).cumprod()
    dd = eq / eq.cummax() - 1
    cg = float((1 + r).prod() ** (252 / len(r)) - 1)
    av = float(r.std() * np.sqrt(252))
    ds = float(np.sqrt((np.minimum(r, 0.0) ** 2).mean()) * np.sqrt(252))
    out = {"total_return": float((1 + r).prod() - 1), "cagr": cg, "ann_vol": av,
           "sharpe": float((r.mean() * 252) / av), "sortino": float((r.mean() * 252) / ds),
           "max_drawdown": float(dd.min()), "calmar": cg / abs(float(dd.min())),
           "win_rate": float((r > 0).mean())}
    if benchmark is not None:
        out["beta"] = float(np.cov(r, benchmark)[0, 1] / np.var(benchmark, ddof=1))
    return pd.Series(out)


plain = performance_report(strat_ret)
assert isinstance(plain, pd.Series), "performance_report should return a pandas Series"
assert list(plain.index) == KEYS, f"without a benchmark the keys must be exactly {KEYS}"
for k in KEYS:
    assert np.isclose(float(plain[k]), float(_ref(strat_ret)[k])), f"the {k} entry is off"

withb = performance_report(strat_ret, benchmark=bench_ret)
assert list(withb.index) == KEYS + ["beta"], "with a benchmark, beta must be appended last"
assert np.isclose(float(withb["beta"]), float(_ref(strat_ret, bench_ret)["beta"])), "beta is off"

costed = performance_report(strat_ret, benchmark=bench_ret, cost_bps=5, positions=position)
ref_costed = _ref(strat_ret, bench_ret, 5, position)
for k in KEYS:
    assert np.isclose(float(costed[k]), float(ref_costed[k])), \
        f"the {k} entry is off once costs are applied - subtract costs BEFORE computing"
assert costed["sharpe"] < withb["sharpe"], "applying costs should lower the Sharpe ratio"

print(f"✅ Correct!  Net of 5 bp: CAGR {float(costed['cagr']):.2%},",
      f"Sharpe {float(costed['sharpe']):.2f}, max DD {float(costed['max_drawdown']):.2%},",
      f"beta {float(costed['beta']):.2f}")

✅ Correct!  Net of 5 bp: CAGR 11.39%, Sharpe 0.84, max DD -21.39%, beta 0.55


## The verdict on our example strategy

Put every number this notebook produced next to each other and write the honest summary.

| Statistic | Value | Reads as |
|---|---|---|
| CAGR | ~11.8% | good in isolation |
| Sharpe | ~0.86 | respectable |
| Sortino | ~1.22 | downside is milder than total vol |
| Max drawdown | ~−21% | tolerable, barely |
| Longest underwater | ~371 trading days | brutal in practice |
| Trade win rate | ~43% | fine, given the payoff |
| Payoff ratio | ~5.0x | this is where the money comes from |
| Round trips | 14 | **too few to conclude anything** |
| Beta | ~0.55 | half a market position |
| Alpha | ~−1.3% | **no skill detected** |
| Information ratio | ~−0.96 | **lost badly to buy-and-hold** |
| Beat benchmark | 1 year in 4 | in the bear market only |

**The summary:** a strategy with acceptable standalone risk statistics, no measurable alpha, that underperformed simply holding the benchmark, over a sample too short to support a firm conclusion either way.

That is a completely ordinary result, and being able to write that sentence about your own work is the actual skill this module teaches. The failure mode is not building a mediocre strategy; everyone does that. The failure mode is building a mediocre strategy and reporting the Sharpe.

Two honest things you could say next: *"the trend filter did its job in the drawdown year, so the idea may be worth keeping as a risk overlay rather than a standalone strategy"*, or *"14 trades is not enough evidence, so the next step is more assets or a longer sample, not more parameter tuning."* Both are useful. "Sharpe 0.86" on its own is not.

## Cheat sheet

**Backtest configuration (QC)**

| Task | Code |
|---|---|
| Realistic broker | `self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE, AccountType.MARGIN)` |
| Benchmark | `self.set_benchmark("SPY")` |
| Market order | `self.market_order(sym, qty)` |
| Limit order | `self.limit_order(sym, qty, limit_price)` |
| Protective stop | `self.stop_market_order(sym, -qty, stop_price)` |
| Go flat | `self.liquidate(sym)` |
| Fill callback | `def on_order_event(self, order_event: OrderEvent):` |
| Was it filled? | `order_event.status == OrderStatus.FILLED` |
| Fill price / fee | `order_event.fill_price`, `order_event.order_fee` |
| Custom chart | `self.plot("Chart", "series", value)` |
| Realised leverage | `self.portfolio.total_holdings_value / self.portfolio.total_portfolio_value` |
| Sample at a fixed time | `self.schedule.on(self.date_rules.every_day(sym), self.time_rules.before_market_close(sym, 5), fn)` |

**Performance statistics (pandas)**

| Statistic | Code |
|---|---|
| Equity curve | `(1 + r).cumprod()` |
| Total return | `(1 + r).prod() - 1` |
| CAGR | `(1 + r).prod() ** (252 / len(r)) - 1` |
| Annualized vol | `r.std() * np.sqrt(252)` |
| Sharpe | `(r.mean() * 252) / (r.std() * np.sqrt(252))` |
| Downside deviation | `np.sqrt((np.minimum(r, 0) ** 2).mean()) * np.sqrt(252)` |
| Sortino | `ann_ret / downside_dev` |
| Drawdown series | `eq / eq.cummax() - 1` |
| Max drawdown | `dd.min()` |
| Underwater runs | `uw.groupby((uw != uw.shift()).cumsum()).sum().max()` |
| Calmar | `cagr / abs(max_dd)` |
| Daily win rate | `(r > 0).mean()` |
| Held mask | `pos.shift(1).fillna(0.0) == 1.0` |
| Trade ids from runs | `(held != held.shift()).cumsum()` |
| Round-trip returns | `r[held].groupby(tid[held]).apply(lambda x: (1 + x).prod() - 1)` |
| Trade win rate | `(trades > 0).mean()` |
| Profit factor | `wins.sum() / abs(losses.sum())` |
| Payoff ratio | `wins.mean() / abs(losses.mean())` |
| Turnover | `pos.diff().abs()` |
| Cost model | `r - pos.diff().abs() * bps / 10_000` |
| Beta | `np.cov(r, b)[0, 1] / np.var(b, ddof=1)` |
| Alpha | `ann_r - beta * ann_b` |
| Active return | `r - benchmark` |
| Tracking error | `active.std() * np.sqrt(252)` |
| Information ratio | `(active.mean() * 252) / tracking_error` |
| Annual returns | `r.resample("YE").apply(lambda x: (1 + x).prod() - 1)` |
| Monthly returns | `r.resample("ME").apply(lambda x: (1 + x).prod() - 1)` |
| Rolling Sharpe | `(r.rolling(252).mean() * 252) / (r.rolling(252).std() * np.sqrt(252))` |
| Sharpe standard error | `np.sqrt((1 + sharpe ** 2 / 2) / n_years)` |

## Stretch goals

Bring these to the next meeting:

1. **Walk-forward analysis.** Split the sample into six blocks. Fit the best `(fast, slow)` on block *k*, then measure it on block *k+1*. How often does the in-sample winner stay a winner? This is the honest version of the parameter sweep, and it is the natural next step after section 8.4.
2. **Plot the diagnostics.** Module 2 covered matplotlib. Draw three charts: the equity curve against the benchmark on a log axis, the drawdown series shaded under zero, and the 252-day rolling Sharpe. Those three panels are what a tear sheet actually is.
3. **Cost curve.** Sweep `cost_bps` from 0 to 50 and plot net CAGR. Where does the strategy stop being viable, and how far is that from a realistic 5–10 bp?
4. **Implement the Probabilistic Sharpe Ratio.** `PSR = Φ((Ŝ − S*) × √(n−1) / √(1 − γ₃Ŝ + (γ₄−1)/4 × Ŝ²))`, where Φ is the normal CDF, γ₃ is the skew of the return distribution (`r.skew()`, how lopsided it is) and γ₄ is its kurtosis (`r.kurt()`, how fat the tails are). Compare it against the naive Sharpe for our strategy. Both statistics are new here. Look them up before you use them.
5. **Extend the snooping experiment.** In section 8.4 we searched 200 zero-edge assets. Repeat it at 10, 50, 200 and 1000 candidates and record the best Sharpe each time. Plot best-Sharpe against number-of-candidates. That curve is the price of searching, and everyone should have seen it once.
6. **On QuantConnect**, run the `RealisticBacktest` algorithm above twice: once with the brokerage model line, once without. Compare Total Fees and Sharpe. How much did realism cost?

## What's next

**Module 7: Statistical Signal Research** moves from evaluating a strategy to *finding* one. You will test whether a series is mean-reverting or trending, build z-scores, run the cointegration test behind pairs trading, and construct a market-neutral spread, with all of this module's statistics available to judge the result.

**Official docs:**
- [Backtesting overview](https://www.quantconnect.com/docs/v2/writing-algorithms/backtesting)
- [Reality modeling](https://www.quantconnect.com/docs/v2/writing-algorithms/reality-modeling/key-concepts)
- [Brokerage models](https://www.quantconnect.com/docs/v2/writing-algorithms/reality-modeling/brokerages/key-concepts)
- [Charting](https://www.quantconnect.com/docs/v2/writing-algorithms/charting)
- [Statistics explained](https://www.quantconnect.com/docs/v2/writing-algorithms/statistics)

*MAT Education · Evaluation & Research · Module 6.*